# Lesson 1 — first table · ตารางแรก

LanceDB ไม่มี server ตารางหนึ่งคือ directory หนึ่ง
เขียนหนึ่งครั้ง ได้ไฟล์สามอย่าง `.txn` · `.manifest` · data fragment
บทนี้สร้างตาราง เพิ่มแถว แล้วเปิด directory ดูว่าเกิดอะไรขึ้นจริง

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


`connect` แค่ชี้ไปที่ folder ยังไม่เกิดไฟล์อะไรทั้งนั้น
ไม่มี socket ไม่มี port สอง process เปิด folder เดียวกันได้

In [2]:
import lancedb

db = lancedb.connect("./data")

ส่ง list ของ dict เข้าไป schema เดาให้เอง
type ที่ได้เป็น Arrow type (`int64` `string`) ไม่ใช่ Python type

In [3]:
rows = [
    {"id": 1, "repo": "lance-indexer", "lang": "ts", "stars": 3},
    {"id": 2, "repo": "session-dream", "lang": "ts", "stars": 7},
    {"id": 3, "repo": "arra-memory-py", "lang": "py", "stars": 1},
]
tbl = db.create_table("repos", data=rows, mode="overwrite")
tbl.schema

[2026-09-10T06:34:24Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/01-first-table/data/repos.lance, it will be created


id: int64
repo: string
lang: string
stars: int64

In [4]:
tbl.to_pandas()

,id,repo,lang,stars
0,1,lance-indexer,ts,3
1,2,session-dream,ts,7
2,3,arra-memory-py,py,1


`where` รับ string หน้าตาเหมือน SQL
เงื่อนไขถูกดันลงไปตอนอ่านไฟล์ ไม่ได้อ่านทั้งหมดขึ้นมาแล้วค่อยกรอง

In [5]:
tbl.search().where("lang = 'ts'").to_pandas()

,id,repo,lang,stars
0,1,lance-indexer,ts,3
1,2,session-dream,ts,7


`add` ไม่แก้ไฟล์เดิม เขียน fragment ใหม่ต่อท้าย แล้วออก manifest ใหม่
version เลยขยับจาก 1 เป็น 2

In [6]:
tbl.add([{"id": 4, "repo": "lanceglass", "lang": "ts", "stars": 2}])
print("count:", tbl.count_rows(), "| version:", tbl.version)

count: 4 | version: 2


เปิด directory ดู
`create` หนึ่งครั้ง `add` หนึ่งครั้ง ก็ควรเห็น txn 2 · manifest 2 · fragment 2

ถ้ารัน notebook นี้ซ้ำ จะเห็นมากกว่านั้น
เพราะ `mode="overwrite"` ไม่ได้ลบของเก่า แค่ออก manifest ใหม่ที่ไม่ชี้ไปหามัน
ไฟล์เก่ายังอยู่ จนกว่าจะสั่ง cleanup เอง (บทที่ 5)

In [7]:
from pathlib import Path

def tree(root: Path, prefix: str = ""):
    kids = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(kids):
        last = i == len(kids) - 1
        print(prefix + ("└── " if last else "├── ") + p.name)
        if p.is_dir():
            tree(p, prefix + ("    " if last else "│   "))

tree(Path("data/repos.lance"))

├── _transactions
│   ├── 0-43eba725-0a5c-4d06-bde1-b520cf38f91d.txn
│   └── 1-0d274fa2-d898-45ce-8df2-ea0a36e19eec.txn
├── _versions
│   ├── 18446744073709551613.manifest
│   ├── 18446744073709551614.manifest
│   └── latest_version_hint.json
└── data
    ├── 001000011110000010110111211ce04edd91d368cd44b00693.lance
    └── 0111111000011100001110113d90084c83b8317b6d4499ccb0.lance
